### University of Virginia
### DS 5110: Big Data Systems
### K-Means Cluster Analysis of Fidelity Fund Returns 
### Last updated: March 11, 2023

## Instructions

In this assignment, you will conduct a k-means cluster analysis on a set of Fidelity mutual funds.  
This helps to group similar funds based on their performance (as opposed to their description, which is typical).  
The outline below will walk you through the required steps.  

This assignment is worth a total of **10 POINTS.**

## Data Details 

The file *fido_returns_funds_on_rows.csv* is the processed data for k-means. Additional details about this file: 
- Each row represents a mutual fund  
- Each column represents a trading day (these are used as features)  
- Each value represents the daily percentage change in price between the current trading day and previous trading day

### Load Modules and Read Data into Spark DataFrame

In [6]:
# Load a Spark session.
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import VectorAssembler

# Load data.
df_fido = spark.read.csv("/standard/ds7200-apt4c/large_datasets/ds5110/07_clustering_and_dim_reduction_mllib/fido_returns_funds_on_rows.csv", header=True, inferSchema=True)

**(VALUE: 2 POINTS) Assemble the Features into a column. 
Show the first five rows of data ONLY for the features column.
(this should make things easier to read)**

In [12]:
# Just use df.columns to get all the features.
assembler = VectorAssembler(inputCols=df_fido.columns, outputCol="features")
dataset=assembler.transform(df_fido)

# Show the first five rows of the features column.
dataset.select("features").show(5)

+--------------------+
|            features|
+--------------------+
|[0.0,0.0,-0.01040...|
|[0.0,0.0,-0.01051...|
|[0.0,0.0,-0.01076...|
|[0.0,8.26105E-4,-...|
|[0.0,8.2815740000...|
+--------------------+
only showing top 5 rows


**(VALUE: 1 POINT) Set up the k-means model and train the model**  
Use parameters: 
- 3 clusters
- maximum of 10 iterations 
- seed=314

In [13]:
# Train the model.
kmeans = KMeans().setK(3).setSeed(314).setMaxIter(10)
model = kmeans.fit(dataset)

26/10/08 21:44:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/10/08 21:45:00 WARN DAGScheduler: Broadcasting large task binary with size 1018.8 KiB


**(VALUE: 2 POINTS) Compute and Print the Silhouette Score**  

In [16]:
# First, compute the predictions.
predictions = model.transform(dataset)

In [19]:
# Show a few of the predictions.
predictions.select("prediction", "features").show(5)

+----------+--------------------+
|prediction|            features|
+----------+--------------------+
|         0|[0.0,0.0,-0.01040...|
|         0|[0.0,0.0,-0.01051...|
|         0|[0.0,0.0,-0.01076...|
|         2|[0.0,8.26105E-4,-...|
|         2|[0.0,8.2815740000...|
+----------+--------------------+
only showing top 5 rows


In [20]:
# Evaluate clustering by computing Silhouette score
evaluator = ClusteringEvaluator()

silhouette = evaluator.evaluate(predictions)
print("Silhouette with squared euclidean distance = " + str(silhouette))

print("Cluster Centers: ")
centers = model.clusterCenters()
print(centers)

Silhouette with squared euclidean distance = 0.4229091846122795
Cluster Centers: 
[array([ 0.        ,  0.00048524, -0.00300457, ...,  0.00115682,
       -0.00934959,  0.00567394], shape=(1731,)), array([ 0.        ,  0.00039189, -0.00063383, ...,  0.00123677,
       -0.00342282, -0.00089758], shape=(1731,)), array([ 0.        , -0.00124346, -0.00686847, ...,  0.00213042,
       -0.01411085,  0.0100746 ], shape=(1731,))]


### For the results above:
- The Silhouette score is ~0.423.  Since the range for the Silhouette is -1 to +1, this value indicates there is definitely some clustering in the data.  
- Each of the three arrays shown represents each of the clusters.
- The values in each array are the calculated centers for each feature.
- Since each feature is a intra-day price change, the first feature has no change since there's no prior day to compare to.


**(VALUE: 2 POINTS) Define a function `kmeans_range()` that does the following:**
- takes an integer representing the lower bound for k
- takes an integer representing the upper bound for k
- take a Spark DataFrame containing training data
- fit K-means with k ranging from lower bound to upper bound, inclusive  
- the other parameters should be the same as earlier 
- for each k, compute the silhouette score
- return a pandas dataframe with columns containing k, silhouette score (each row holds the score for given k)

**(VALUE: 1 POINT) Call `kmeans_range` to compute K-means with clusters ranging from 2 to 10 inclusive, printing the resulting dataframe.**

**(VALUE: 1 POINT) Produce a plot with cluster numbers k on the x-axis, sihouette scores on the y-axis**

**(VALUE: 1 POINT) Based on how the silhouette score is calculated, what is its time complexity? (e.g., O(log n))**  
You can find the definition of the silhouette score in the lecture notes, for example. 